# 2026-09-30 005930 축 확인 (T-002 원인 1)

- 목적: 체결 점이 맨 위·맨 아래에 붙어 보이는 원인을 눈으로 확인한다.
- 원인: `plot()`이 tick을 왼쪽 축에, trade를 오른쪽 축에 그린다. 두 축은 눈금이 다르다.
- 근거: `out/tasks/T-002/0930_시세_체결_불일치_분석.md` 3절, 4절.
- 셀을 위에서 아래로 차례대로 실행한다.

## 0. 경로 설정

- `alpha`를 import할 수 없으면 이 셀이 위 폴더에서 `src/`를 찾아 `sys.path`에 넣는다.
- Docker(`/app`)에서는 `pip install -e .`로 이미 import가 된다. 그때 이 셀은 아무것도 바꾸지 않는다.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

try:
    import alpha  # noqa: F401
except ImportError:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "alpha").is_dir():
            sys.path.insert(0, str(p / "src"))
            break
    import alpha  # noqa: F401

print("alpha:", alpha.__file__)

## 1. 데이터 읽기

- 원본 노트북 `notebook/2026-10-03-010_trading.ipynb` 셀 1~3, 6과 같다.
- `trades.df`에 2행(10:28:01 entry 273,500, 10:30:01 exit 272,500)이 나오면 맞다.

In [ ]:
import pandas as pd
from alpha.analysist import datastore

ds = datastore.DataStore(live=False)

s = "2026-09-30"
e = "2026-09-30"
trades = ds.trades(start=s, end=e)
tick = ds.ticks(symbol="005930", start=s, end=e)

trades.df

## 2. 원래 그림 (원본 셀 7)

- 체결 점이 오른쪽 축에 따로 그려진다. 오른쪽 눈금은 점 2개만으로 정해진다.
- 그래서 진입 점은 맨 위에, 청산 점은 맨 아래에 붙어 보인다.

In [ ]:
(tick.resample_frame("s")
     .time_frame(mode="s", start=s + " 09:00:00", end=e + " 15:30:00")
     .columns("price")
     .add(trades)
     .plot(kind={"trade": "mark"}))

## 3. 같은 축 그림

- `layout=[(["tick", "trade"], [])]`로 tick과 trade를 왼쪽 축 하나에 그린다. 오른쪽 축은 없다.
- 체결 점 273,500(10:28:01 진입)과 272,500(10:30:01 청산)이 tick 선 위에 찍히면 맞다.

In [ ]:
(tick.resample_frame("s")
     .time_frame(mode="s", start=s + " 09:00:00", end=e + " 15:30:00")
     .columns("price")
     .add(trades)
     .plot(layout=[(["tick", "trade"], [])], kind={"trade": "mark"}))

## 4. 확대 그림 (10:25~10:35)

- tick의 price, bid, ask와 체결 점을 같은 축에 그린다.
- 진입 점(10:28:01)이 ask 선 위에, 청산 점(10:30:01)이 bid 선 위에 있으면 맞다.
- `dropna=True`로 빈 초를 건너 선을 잇는다. 끊으면 tick이 드문 구간의 선이 보이지 않는다.

In [ ]:
(tick.resample_frame("s")
     .time_frame(mode="s", start=s + " 10:25:00", end=e + " 10:35:00")
     .columns(["price", "bid", "ask"])
     .add(trades)
     .plot(layout=[(["tick", "trade"], [])], kind={"trade": "mark"}, dropna=True))

## 5. 숫자 확인

- 체결 시각마다 그 시각 직전(`dt < 체결 시각`)의 마지막 tick을 찾는다. 리샘플하지 않은 원본 tick을 쓴다.
- 진입은 `체결가 == ask`, 청산은 `체결가 == bid`이면 맞다 (SimBroker 시장가 규칙, T-002 4.1절).

In [ ]:
tk = tick.df.sort_index()
rows = []
for t, tr in trades.df.sort_index().iterrows():
    t = pd.Timestamp(t)
    before = tk[tk.index < t]
    last = before.iloc[-1]
    rows.append({
        "체결 시각": t,
        "side": tr["side"],
        "체결가": tr["price"],
        "직전 tick 시각": before.index[-1],
        "tick price": last["price"],
        "bid": last["bid"],
        "ask": last["ask"],
        "체결가 == ask": tr["price"] == last["ask"],
        "체결가 == bid": tr["price"] == last["bid"],
    })
pd.DataFrame(rows)

- 아래 표는 체결 시각 앞 3초부터 체결 시각까지의 원본 tick이다. 위 표의 값을 행 단위로 본다.

In [ ]:
for t in trades.df.index:
    t = pd.Timestamp(t)
    print("체결 시각:", t)
    display(tk.loc[t - pd.Timedelta(seconds=3): t, ["symbol", "price", "bid", "ask"]])